In [ ]:
import numpy as np
import pandas as pd
import string
from string import digits

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Embedding,
    Dense,
    Layer,
    Concatenate
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Load Dataset
DATA_PATH = "/content/Hindi_English_Truncated_Corpus.csv"

lines = pd.read_csv(DATA_PATH, encoding="utf-8", on_bad_lines="skip")

print("Original dataset size:", len(lines))


In [ ]:
# Select TED Dataset
lines = (
    lines[lines["source"] == "ted"]
    [["english_sentence", "hindi_sentence"]]
    .dropna()
    .drop_duplicates()
)

lines = lines.sample(
    n=25000,
    random_state=42
).reset_index(drop=True)

print("Dataset size:", len(lines))


In [ ]:
# Text Cleaning

def clean_text(text):

    exclude = set(string.punctuation)

    text = "".join(
        ch
        for ch in text
        if ch not in exclude
    )

    text = text.translate(
        str.maketrans("", "", digits)
    )

    return text.strip().lower()


lines["english_sentence"] = (
    lines["english_sentence"]
    .apply(clean_text)
)

lines["hindi_sentence"] = (
    lines["hindi_sentence"]
    .apply(clean_text)
)


In [ ]:
# Add Start / End Tokens

lines["hindi_sentence"] = (
    lines["hindi_sentence"]
    .apply(
        lambda x:
        "start_ " + x + " _end"
    )
)


In [ ]:
# Tokenization
eng_tokenizer = Tokenizer()

eng_tokenizer.fit_on_texts(
    lines["english_sentence"]
)

eng_seq = eng_tokenizer.texts_to_sequences(
    lines["english_sentence"]
)


hin_tokenizer = Tokenizer(
    filters=""
)

hin_tokenizer.fit_on_texts(
    lines["hindi_sentence"]
)

hin_seq = hin_tokenizer.texts_to_sequences(
    lines["hindi_sentence"]
)


In [ ]:
# Vocabulary Sizes
eng_vocab_size = (
    len(eng_tokenizer.word_index) + 1
)

hin_vocab_size = (
    len(hin_tokenizer.word_index) + 1
)

print("English vocabulary:", eng_vocab_size)
print("Hindi vocabulary:", hin_vocab_size)


In [ ]:
# Maximum Sequence Lengths

max_eng_len = max(
    len(seq)
    for seq in eng_seq
)

max_hin_len = max(
    len(seq)
    for seq in hin_seq
)

print("Max English length:", max_eng_len)
print("Max Hindi length:", max_hin_len)


In [ ]:
# Padding

eng_seq = [seq for seq in eng_seq if len(seq) > 0]
hin_seq = [seq for seq in hin_seq if len(seq) > 0]

encoder_input = pad_sequences(
    eng_seq,
    maxlen=max_eng_len,
    padding="post"
)

decoder_full = pad_sequences(
    hin_seq,
    maxlen=max_hin_len,
    padding="post"
)


In [ ]:
# Decoder Input / Target
# Example:
# Hindi:
# start_   I   am   happy   _end
# decoder_input:
# start_   I   am   happy
# decoder_target:
# I        am   happy   _end

decoder_input = decoder_full[:, :-1]

decoder_target = decoder_full[:, 1:]

decoder_target = np.expand_dims(
    decoder_target,
    axis=-1
)

decoder_max_len = decoder_input.shape[1]


In [ ]:
# Ignore Padding in Loss
sample_weights = (
    decoder_target[:, :, 0] != 0
).astype("float32")


In [ ]:
# Hyperparameters
latent_dim = 256

In [ ]:
# Encoder
encoder_inputs = Input(
    shape=(None,),
    name="encoder_inputs"
)


encoder_embedding_layer = Embedding(
    input_dim=eng_vocab_size,
    output_dim=latent_dim,
    mask_zero=False,
    name="encoder_embedding"
)

enc_emb = encoder_embedding_layer(
    encoder_inputs
)


encoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="encoder_lstm"
)


encoder_outputs, state_h, state_c = encoder_lstm(
    enc_emb
)


encoder_states = [
    state_h,
    state_c
]


In [ ]:
# Decoder

decoder_inputs = Input(
    shape=(None,),
    name="decoder_inputs"
)


decoder_embedding_layer = Embedding(
    input_dim=hin_vocab_size,
    output_dim=latent_dim,
    mask_zero=False,
    name="decoder_embedding"
)
dec_emb = decoder_embedding_layer(
    decoder_inputs
)


decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)


decoder_outputs, _, _ = decoder_lstm(
    dec_emb,
    initial_state=encoder_states
)


In [ ]:
class BahdanauAttention(Layer):

    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)

        self.units = units

        self.Wq = Dense(
            units,
            use_bias=False,
            name="Wq"
        )

        self.Wk = Dense(
            units,
            use_bias=False,
            name="Wk"
        )

        self.V = Dense(
            1,
            use_bias=False,
            name="V"
        )

    def call(self, query, values):

        # query:
        # (batch_size, decoder_len, latent_dim)
        #
        # values:
        # (batch_size, encoder_len, latent_dim)

        query_projection = self.Wq(query)

        value_projection = self.Wk(values)

        # ----------------------------------------------------
        # Expand dimensions so every decoder timestep
        # can compare itself with every encoder timestep.
        #
        # query_projection:
        #
        # (batch, decoder_len, 1, units)
        #
        # value_projection:
        #
        # (batch, 1, encoder_len, units)
        # ----------------------------------------------------

        query_projection = tf.expand_dims(
            query_projection,
            axis=2
        )

        value_projection = tf.expand_dims(
            value_projection,
            axis=1
        )

        # ----------------------------------------------------
        # Additive attention score
        #
        # score(t, i) =
        # V(tanh(Wq * query_t + Wk * key_i))
        # ----------------------------------------------------

        score = self.V(
            tf.nn.tanh(
                query_projection +
                value_projection
            )
        )

        # score:
        #
        # (batch, decoder_len, encoder_len, 1)

        score = tf.squeeze(
            score,
            axis=-1
        )

        # score:
        #
        # (batch, decoder_len, encoder_len)

        attention_weights = tf.nn.softmax(
            score,
            axis=-1
        )

        # ----------------------------------------------------
        # Context vector
        #
        # context_t =
        # sum_i attention(t,i) * encoder_output_i
        # ----------------------------------------------------

        context = tf.matmul(
            attention_weights,
            values
        )

        # context:
        #
        # (batch, decoder_len, latent_dim)

        return context, attention_weights

In [ ]:
# Attention
attention_layer = BahdanauAttention(
    latent_dim,
    name="attention"
)

context_vectors, attention_scores = attention_layer(
    decoder_outputs,
    encoder_outputs
)

In [ ]:
# Combine Decoder Output + Context
decoder_combined_context = Concatenate(
    axis=-1,
    name="decoder_context"
)(
    [
        decoder_outputs,
        context_vectors
    ]
)



In [ ]:
# Output Layer
decoder_dense = Dense(
    hin_vocab_size,
    activation="softmax",
    name="decoder_dense"
)


decoder_outputs_final = decoder_dense(
    decoder_combined_context
)




In [ ]:
# Training Model
model = Model(
    [
        encoder_inputs,
        decoder_inputs
    ],
    decoder_outputs_final,
    name="seq2seq_attention_model"
)


In [ ]:
# Compile
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy"
)



In [ ]:
# Model Summary

model.summary()


In [ ]:
# Training
history = model.fit(
    [
        encoder_input,
        decoder_input
    ],
    decoder_target,
    sample_weight=sample_weights,
    batch_size=64,
    epochs=20,
    validation_split=0.2
)


In [ ]:
# Reverse Word Index

reverse_eng = {
    index: word
    for word, index
    in eng_tokenizer.word_index.items()
}

reverse_hin = {
    index: word
    for word, index
    in hin_tokenizer.word_index.items()
}


In [ ]:
# Encoder Inference Model

encoder_model_inf = Model(
    encoder_inputs,
    [
        encoder_outputs,
        state_h,
        state_c
    ],
    name="encoder_inference_model"
)


In [ ]:
#  Decoder Inference Inputs

decoder_state_input_h = Input(
    shape=(latent_dim,),
    name="decoder_state_input_h"
)

decoder_state_input_c = Input(
    shape=(latent_dim,),
    name="decoder_state_input_c"
)

encoder_outputs_input = Input(
    shape=(None, latent_dim),
    name="encoder_outputs_input"
)


In [ ]:
# Decoder Inference Embedding

dec_inf_emb = decoder_embedding_layer(
    decoder_inputs
)


In [ ]:
#  Decoder LSTM - One Step

dec_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    dec_inf_emb,
    initial_state=[
        decoder_state_input_h,
        decoder_state_input_c
    ]
)



In [ ]:
# Attention During Inference

context_inf, attention_scores_inf = attention_layer(
    dec_outputs_inf,
    encoder_outputs_input
)


In [ ]:
# Combine Decoder Output + Context

dec_combined_inf = Concatenate(
    axis=-1,
    name="decoder_context_inf"
)(
    [
        dec_outputs_inf,
        context_inf
    ]
)

In [ ]:
#  Predict Next Token

decoder_outputs_inf = decoder_dense(
    dec_combined_inf
)


In [ ]:
# Decoder Inference Model

decoder_model_inf = Model(
    [
        decoder_inputs,
        decoder_state_input_h,
        decoder_state_input_c,
        encoder_outputs_input
    ],
    [
        decoder_outputs_inf,
        state_h_inf,
        state_c_inf,
        attention_scores_inf
    ],
    name="decoder_inference_model"
)


In [ ]:
# Translation Function

def translate(sentence):

    # --------------------------------------------------------
    # Clean input
    # --------------------------------------------------------

    sentence = clean_text(sentence)


    # --------------------------------------------------------
    # Tokenize English sentence
    # --------------------------------------------------------

    seq = eng_tokenizer.texts_to_sequences(
        [sentence]
    )


    # --------------------------------------------------------
    # Padding
    # --------------------------------------------------------

    padded = pad_sequences(
        seq,
        maxlen=max_eng_len,
        padding="post"
    )


    # --------------------------------------------------------
    # Encoder
    # --------------------------------------------------------

    encoder_out, h, c = (
        encoder_model_inf.predict(
            padded,
            verbose=0
        )
    )


    # --------------------------------------------------------
    # Start token
    # --------------------------------------------------------

    target_seq = np.zeros(
        (1, 1),
        dtype="int32"
    )

    start_token_index = (
        hin_tokenizer.word_index["start_"]
    )

    target_seq[0, 0] = (
        start_token_index
    )


    # --------------------------------------------------------
    # Storage
    # --------------------------------------------------------

    decoded_words = []

    attention_history = []


    # ========================================================
    # Autoregressive Decoding
    # ========================================================

    for _ in range(decoder_max_len):

        output, h, c, attention = (
            decoder_model_inf.predict(
                [
                    target_seq,
                    h,
                    c,
                    encoder_out
                ],
                verbose=0
            )
        )


        # ----------------------------------------------------
        # Find predicted token
        # ----------------------------------------------------

        token_index = np.argmax(
            output[0, -1, :]
        )


        # ----------------------------------------------------
        # Token ID -> Word
        # ----------------------------------------------------

        word = reverse_hin.get(
            token_index,
            ""
        )


        # ----------------------------------------------------
        # Save attention
        # ----------------------------------------------------

        attention_history.append(
            attention[0, 0, :]
        )


        # ----------------------------------------------------
        # Stop
        # ----------------------------------------------------

        if word == "_end":
            break

        if word == "":
            break


        # ----------------------------------------------------
        # Save generated word
        # ----------------------------------------------------

        decoded_words.append(
            word
        )


        # ----------------------------------------------------
        # Previous prediction becomes
        # next decoder input
        # ----------------------------------------------------

        target_seq = np.zeros(
            (1, 1),
            dtype="int32"
        )

        target_seq[0, 0] = token_index


    return (
        " ".join(decoded_words),
        attention_history
    )


In [ ]:
# Test Translation

test_sentence = "I love you"

translation, attention = translate(
    test_sentence
)


print("\n====================================")
print("Translation")
print("====================================")

print("English:")
print(test_sentence)

print("\nHindi:")
print(translation)



In [ ]:
# Display Attention

english_words = test_sentence.split()

hindi_words = translation.split()


print("\n====================================")
print("Attention")
print("====================================")


for i, attention_vector in enumerate(
    attention
):

    if i >= len(hindi_words):
        break

    predicted_word = hindi_words[i]

    print(
        f"\nGenerated word: {predicted_word}"
    )

    print(
        "Attention weights:"
    )

    for eng_word, weight in zip(
        english_words,
        attention_vector[:len(english_words)]
    ):

        print(
            f"    {eng_word:10s} -> {weight:.4f}"
        )


In [ ]:
# Attention Matrix

if len(attention) > 0:

    attention_matrix = np.array(
        attention
    )

    attention_matrix = attention_matrix[
        :len(hindi_words),
        :len(english_words)
    ]


    print("\n====================================")
    print("Attention Matrix")
    print("====================================")

    print(
        attention_matrix
    )